<a href="https://colab.research.google.com/github/Farida2025/Api_assignment/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22Untitled2_ipynb%22%22%22%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#nirs

## 1. Install libraries

In [30]:
!pip -q install networkx folium pandas numpy requests tqdm matplotlib


## 2. Imports and settings


In [31]:
import os, json, math
import requests
import numpy as np
import pandas as pd
import networkx as nx
from tqdm import tqdm
import matplotlib.pyplot as plt

## 3. Authentication

In [32]:
import requests
import os
from google.colab import userdata

BASE = "http://busstop.cts.gov.kz:8085/api/v1"

try:
    USERNAME = userdata.get('CTS_USERNAME')
    PASSWORD = userdata.get('CTS_PASSWORD')
except userdata.SecretNotFoundError:
    print("Ошибка: Секреты не найдены. Проверьте вкладку с ключом в левом меню.")

r = requests.post(
    f"{BASE}/auth/login",
    json={"username": USERNAME, "password": PASSWORD},
    timeout=30
)
r.raise_for_status()

token = r.json()["accessToken"]
headers = {"Authorization": f"Bearer {token}", "Accept": "application/json"}

print("OK token length:", len(token))

HTTPError: 403 Client Error:  for url: http://busstop.cts.gov.kz:8085/api/v1/auth/login

## 4. Download routes and stops


In [ ]:
routes = requests.get(f"{BASE}/routes", headers=headers, timeout=30).json()
stops = requests.get(f"{BASE}/bus-stop/all", headers=headers, timeout=30).json()

print("routes:", len(routes))
print("stops:", len(stops))

os.makedirs("data_raw", exist_ok=True)

with open("data_raw/routes.json", "w", encoding="utf-8") as f:
    json.dump(routes, f, ensure_ascii=False)

with open("data_raw/stops.json", "w", encoding="utf-8") as f:
    json.dump(stops, f, ensure_ascii=False)


## 5. Routes WITH bus stops


In [ ]:
def get_route_with_stops(route_id: str):
    url = f"{BASE}/routes/{route_id}/with-bus-stops"
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    return resp.json()

N = min(800, len(routes))

routes_with_stops = []

for rt in tqdm(routes[:N]):
    rid = rt["id"]
    try:
        rw = get_route_with_stops(rid)
        routes_with_stops.append(rw)
    except Exception as e:
        print("skip", rid, "err:", str(e)[:120])

print("loaded routes_with_stops:", len(routes_with_stops))

with open("data_raw/routes_with_stops.json", "w", encoding="utf-8") as f:
    json.dump(routes_with_stops, f, ensure_ascii=False)


In [ ]:
def extract_stop_ids_fw_bw(route_obj):
    """
    Returns (fw_ids, bw_ids) — ordered lists of stop IDs
    from busStopsForward / busStopsBackward.
    """
    fw = route_obj.get("busStopsForward", [])
    bw = route_obj.get("busStopsBackward", [])

    def to_ids(lst):
        ids = []
        if not isinstance(lst, list):
            return ids
        for x in lst:
            if not isinstance(x, dict):
                continue
            if "id" in x:
                ids.append(x["id"])
            elif "busStopId" in x:
                ids.append(x["busStopId"])
        return ids

    fw_ids = to_ids(fw)
    bw_ids = to_ids(bw)
    return fw_ids, bw_ids


In [ ]:
stops_df = pd.DataFrame(stops)

def pick_latlon(obj):
    lat = obj.get("latitude", obj.get("lat"))
    lon = obj.get("longitude", obj.get("lon", obj.get("lng")))
    return (float(lat), float(lon)) if lat is not None and lon is not None else (None, None)

stop_id_to_name = {}
stop_id_to_coord = {}

for s in stops:
    sid = s.get("id")
    name = s.get("name", s.get("title", f"stop_{sid}"))
    lat, lon = pick_latlon(s)
    if sid is None or lat is None:
        continue
    stop_id_to_name[sid] = name
    stop_id_to_coord[sid] = (lat, lon)

print("stops loaded:", len(stop_id_to_coord))


##6. Osrm

In [ ]:
import os, json, time, requests

OSRM_URL = "https://router.project-osrm.org"
OSRM_PROFILE = "driving"

os.makedirs("cache_osrm", exist_ok=True)
CACHE_PATH = "cache_osrm/route_cache.json"

if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, "r", encoding="utf-8") as f:
        OSRM_CACHE = json.load(f)
else:
    OSRM_CACHE = {}

def cache_key(a_lat, a_lon, b_lat, b_lon):
    a_lat, a_lon = round(float(a_lat), 6), round(float(a_lon), 6)
    b_lat, b_lon = round(float(b_lat), 6), round(float(b_lon), 6)
    return f"{a_lat},{a_lon}->{b_lat},{b_lon}"

def osrm_route_duration_distance(a_lat, a_lon, b_lat, b_lon, retries=2):
    key = cache_key(a_lat, a_lon, b_lat, b_lon)
    if key in OSRM_CACHE:
        d = OSRM_CACHE[key]
        return d["duration_s"], d["distance_m"]

    coords = f"{a_lon},{a_lat};{b_lon},{b_lat}"
    url = f"{OSRM_URL}/route/v1/{OSRM_PROFILE}/{coords}"
    params = {"overview": "false", "alternatives": "false", "steps": "false"}

    for _ in range(retries + 1):
        try:
            r = requests.get(url, params=params, timeout=20)
            if r.status_code == 429:
                time.sleep(1.0)
                continue
            r.raise_for_status()
            data = r.json()

            if data.get("code") != "Ok" or not data.get("routes"):
                break

            route = data["routes"][0]
            duration_s = float(route["duration"])
            distance_m = float(route["distance"])

            OSRM_CACHE[key] = {"duration_s": duration_s, "distance_m": distance_m}
            return duration_s, distance_m
        except Exception:
            time.sleep(0.3)

    return None, None

def save_osrm_cache():
    with open(CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(OSRM_CACHE, f, ensure_ascii=False)

In [ ]:
import networkx as nx
import numpy as np
from collections import defaultdict
from tqdm import tqdm
import time

G_stop_osrm = nx.DiGraph()

edge_routes = defaultdict(set)
edge_dirs   = defaultdict(set)
edge_durs   = defaultdict(list)
edge_dists  = defaultdict(list)

def process_seq(seq, direction, rid):
    if not seq or len(seq) < 2:
        return
    for a, b in zip(seq, seq[1:]):
        if a not in stop_id_to_coord or b not in stop_id_to_coord or a == b:
            continue

        a_lat, a_lon = stop_id_to_coord[a]
        b_lat, b_lon = stop_id_to_coord[b]

        dur, dist = osrm_route_duration_distance(a_lat, a_lon, b_lat, b_lon, retries=1)
        if dur is None or dist is None or dur <= 0 or dist <= 0:
            continue

        edge_routes[(a, b)].add(rid)
        edge_dirs[(a, b)].add(direction)
        edge_durs[(a, b)].append(float(dur))
        edge_dists[(a, b)].append(float(dist))

        time.sleep(0.01)

used_routes = routes_with_stops[:100]

for r in tqdm(used_routes):
    rid = r.get("id")
    fw_ids, bw_ids = extract_stop_ids_fw_bw(r)
    process_seq(fw_ids, "F", rid)
    process_seq(bw_ids, "B", rid)

for (a, b) in edge_routes.keys():
    t_med = float(np.median(edge_durs[(a, b)]))
    d_med = float(np.median(edge_dists[(a, b)]))
    speed_kmh = (d_med / t_med) * 3.6

    G_stop_osrm.add_edge(
        a, b,
        routes=set(edge_routes[(a, b)]),
        dirs=set(edge_dirs[(a, b)]),
        load=len(edge_routes[(a, b)]),
        travel_time_s=t_med,
        distance_m=d_med,
        speed_kmh=speed_kmh
    )

print("G_stop_osrm nodes:", G_stop_osrm.number_of_nodes())
print("G_stop_osrm edges:", G_stop_osrm.number_of_edges())

save_osrm_cache()
print("OSRM cache size:", len(OSRM_CACHE))

In [ ]:
a, b, d = next(iter(G_stop_osrm.edges(data=True)))
print("Example edge:", a, "->", b)
print("distance_m:", d["distance_m"])
print("travel_time_s:", d["travel_time_s"])
print("speed_kmh:", d["speed_kmh"])
print("load:", d["load"])

In [ ]:

for s in G_stop_osrm.nodes():
    rset = set()
    for _,_,d in G_stop_osrm.in_edges(s, data=True):
        rset |= d["routes"]
    for _,_,d in G_stop_osrm.out_edges(s, data=True):
        rset |= d["routes"]
    G_stop_osrm.nodes[s]["routes_cnt"] = len(rset)

print("routes_cnt added to nodes")

In [ ]:
import pandas as pd

df_stop = pd.DataFrame([
    {
        "stop_id": sid,
        "name": stop_id_to_name.get(sid, str(sid)),
        "lat": stop_id_to_coord[sid][0],
        "lon": stop_id_to_coord[sid][1],
        "routes_cnt": G_stop_osrm.nodes[sid]["routes_cnt"],
        "degree": G_stop_osrm.degree(sid)
    }
    for sid in G_stop_osrm.nodes()
])
df_stop.head()

In [ ]:
import networkx as nx

k = min(800, G_stop_osrm.number_of_nodes())
stop_b = nx.betweenness_centrality(G_stop_osrm,
                                   k=k,
                                   seed=42,
                                   weight="travel_time_s")

df_stop["betw"] = df_stop["stop_id"].map(lambda x: stop_b.get(x, 0.0))
df_stop["routes_n"] = df_stop["routes_cnt"] / max(1e-9, df_stop["routes_cnt"].max())
df_stop["betw_n"] = df_stop["betw"] / max(1e-9, df_stop["betw"].max())
df_stop["bottleneck_score"] = df_stop["routes_n"] * df_stop["betw_n"]

df_stop.sort_values("bottleneck_score", ascending=False).head(20)

## 7. Download route geometry (map)


In [ ]:
def get_route_map(route_id: str):
    url = f"{BASE}/routes/{route_id}/map"
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    return resp.json()

N = 300
route_maps = []

for rt in tqdm(routes[:N]):
    rid = rt["id"] if isinstance(rt, dict) and "id" in rt else rt
    try:
        rm = get_route_map(rid)
        route_maps.append({"id": rid, "map": rm})
    except Exception as e:
        print("skip route", rid, "err", str(e)[:120])

with open("data_raw/route_maps.json", "w", encoding="utf-8") as f:
    json.dump(route_maps, f, ensure_ascii=False)

print("loaded route_maps:", len(route_maps))


## 8. Extract coordinates from route_map


In [ ]:
def extract_coords(route_map_obj, direction="forward"):
    if not isinstance(route_map_obj, dict):
        return None

    path = route_map_obj.get("path")
    if not isinstance(path, dict):
        return None

    pts = path.get(direction)
    if not isinstance(pts, list) or len(pts) < 2:
        return None

    coords = []
    for p in pts:
        if not isinstance(p, dict):
            continue

        lat = p.get("latitude", p.get("lat"))
        lon = p.get("longitude", p.get("lon", p.get("lng")))

        if lat is None or lon is None:
            continue

        coords.append((float(lat), float(lon)))

    return coords if len(coords) >= 2 else None


In [ ]:
sample = route_maps[0]["map"]
c1 = extract_coords(sample, "forward")
c2 = extract_coords(sample, "backward")
print("forward:", len(c1) if c1 else None, "backward:", len(c2) if c2 else None)

## 9. Build segment graph (NetworkX)


In [ ]:
import math
import networkx as nx
from tqdm import tqdm

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))

def norm_point(lat, lon, ndigits=5):
    return (round(float(lat), ndigits), round(float(lon), ndigits))

G = nx.DiGraph()
MAX_POINTS_PER_ROUTE = 3000
ROUND_DIGITS = 5

def add_route_to_graph(G, coords, route_id):
    if not coords or len(coords) < 2:
        return

    step = max(1, len(coords) // MAX_POINTS_PER_ROUTE)
    coords = coords[::step]

    for i in range(len(coords) - 1):
        u = norm_point(coords[i][0], coords[i][1], ROUND_DIGITS)
        v = norm_point(coords[i + 1][0], coords[i + 1][1], ROUND_DIGITS)

        if u == v:
            continue

        dist = haversine(u[0], u[1], v[0], v[1])
        if dist <= 0:
            continue

        if G.has_edge(u, v):
            G[u][v]["routes"].add(route_id)
        else:
            G.add_edge(u, v, length_m=dist, routes=set([route_id]))

for item in tqdm(route_maps):
    rid = item["id"]
    rm = item["map"]
    add_route_to_graph(G, extract_coords(rm, "forward"), rid)
    add_route_to_graph(G, extract_coords(rm, "backward"), rid)

for u, v in G.edges():
    G[u][v]["load"] = len(G[u][v]["routes"])

print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())


In [ ]:
weakly_ccs = list(nx.weakly_connected_components(G))
largest_cc_nodes = max(weakly_ccs, key=len)
G_main = G.subgraph(largest_cc_nodes).copy()

print("Main component nodes:", G_main.number_of_nodes(), "edges:", G_main.number_of_edges())

loads = [G_main[u][v]["load"] for u, v in G_main.edges()]
lengths = [G_main[u][v]["length_m"] for u, v in G_main.edges()]

print("Edges load: min/mean/max =", min(loads), f"{sum(loads)/len(loads):.2f}", max(loads))
shared = sum(1 for x in loads if x >= 2)
print("Edges shared by 2+ routes:", shared, f"({shared/len(loads)*100:.1f}%)")

## 10. G_stop_osrm


In [ ]:
import pandas as pd
import networkx as nx
import numpy as np

k_edges = min(1200, G_stop_osrm.number_of_nodes())
edge_b = nx.edge_betweenness_centrality(G_stop_osrm,
                                        k=k_edges,
                                        seed=42,
                                        weight="travel_time_s")

rows = []
for (a, b), betw in edge_b.items():
    d = G_stop_osrm[a][b]
    a_lat, a_lon = stop_id_to_coord[a]
    b_lat, b_lon = stop_id_to_coord[b]
    rows.append({
        "a": a, "b": b,
        "a_lat": a_lat, "a_lon": a_lon,
        "b_lat": b_lat, "b_lon": b_lon,
        "load": d["load"],
        "betw": betw,
        "distance_m": d["distance_m"],
        "travel_time_s": d["travel_time_s"],
        "speed_kmh": d["speed_kmh"],
    })

df_edges_osrm = pd.DataFrame(rows)

df_edges_osrm["load_n"] = df_edges_osrm["load"] / max(1e-9, df_edges_osrm["load"].max())
df_edges_osrm["betw_n"] = df_edges_osrm["betw"] / max(1e-9, df_edges_osrm["betw"].max())
df_edges_osrm["time_n"] = df_edges_osrm["travel_time_s"] / max(1e-9, df_edges_osrm["travel_time_s"].max())

df_edges_osrm["risk"] = (df_edges_osrm["load_n"] *
                        df_edges_osrm["betw_n"] *
                        df_edges_osrm["time_n"])

df_edges_osrm.sort_values("risk", ascending=False).head(20)

In [ ]:
import os
os.makedirs("results", exist_ok=True)

In [ ]:
df_edges_osrm.to_csv("results/osrm_edges_with_risk.csv", index=False)

## 11. GNN


In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q torch-geometric

In [ ]:
import torch
from torch_geometric.nn import GCNConv

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

nodes_list = list(G_stop_osrm.nodes())
node_to_idx = {n:i for i,n in enumerate(nodes_list)}

risk_max = df_edges_osrm["risk"].max()
risk_map = {(row["a"], row["b"]): float(row["risk"]/risk_max) for _, row in df_edges_osrm.iterrows()}

edge_index = []
edge_attr = []
y = []

for a, b, data in G_stop_osrm.edges(data=True):
    edge_index.append([node_to_idx[a], node_to_idx[b]])
    edge_attr.append([
        data["distance_m"]/1000.0,
        data["travel_time_s"]/300.0,
        data["speed_kmh"]/60.0,
        data["load"]/20.0
    ])
    y.append(risk_map.get((a,b), 0.0))

edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
edge_attr = torch.tensor(edge_attr, dtype=torch.float)
y = torch.tensor(y, dtype=torch.float)

node_features = []
for n in nodes_list:
    lat, lon = stop_id_to_coord[n]
    node_features.append([
        G_stop_osrm.degree(n)/10.0,
        (lat - 51.1)*10,
        (lon - 71.4)*10
    ])
x = torch.tensor(node_features, dtype=torch.float)

class TransportGNN(nn.Module):
    def __init__(self, n_node_feat=3, n_edge_feat=4, hidden=32):
        super().__init__()
        self.conv1 = GCNConv(n_node_feat, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden*2 + n_edge_feat, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
            nn.Sigmoid()
        )

    def forward(self, x, edge_index, edge_attr):
        h = F.relu(self.conv1(x, edge_index))
        h = F.relu(self.conv2(h, edge_index))
        row, col = edge_index
        edge_input = torch.cat([h[row], h[col], edge_attr], dim=-1)
        return self.edge_mlp(edge_input).squeeze()

model = TransportGNN()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
crit = nn.HuberLoss()

model.train()
for epoch in range(150):
    opt.zero_grad()
    pred = model(x, edge_index, edge_attr)
    loss = crit(pred, y)
    loss.backward()
    opt.step()
    if epoch % 30 == 0:
        print(epoch, loss.item())

model.eval()
with torch.no_grad():
    preds = model(x, edge_index, edge_attr).cpu().numpy()

edges_list = [(a,b) for a,b,_ in G_stop_osrm.edges(data=True)]
pred_map = {edges_list[i]: float(preds[i]) for i in range(len(edges_list))}

df_edges_osrm["gnn_score"] = df_edges_osrm.apply(lambda r: pred_map.get((r["a"], r["b"]), 0.0), axis=1)
df_edges_osrm["final_criticality_index"] = 0.5*df_edges_osrm["risk"] + 0.5*df_edges_osrm["gnn_score"]

df_edges_osrm.sort_values("final_criticality_index", ascending=False).head(20)

##12. Baseline comparison

In [ ]:
import pandas as pd
import networkx as nx

def largest_cc_ratio(Gx):
    ccs = list(nx.weakly_connected_components(Gx))
    m = max((len(c) for c in ccs), default=0)
    return m / max(1, Gx.number_of_nodes())

def robustness_metrics_after_removal(G_base, edges_ranked, K):

    H = G_base.copy()
    for (a, b) in edges_ranked[:min(K, len(edges_ranked))]:
        if H.has_edge(a, b):
            H.remove_edge(a, b)

    return {
        "R(K)": largest_cc_ratio(H),
        "Components(K)": nx.number_weakly_connected_components(H)
    }

required_cols = {"a","b","distance_m","final_criticality_index"}
missing = required_cols - set(df_edges_osrm.columns)
if missing:
    raise ValueError(f"df_edges_osrm is missing columns: {missing}")

rank_osm_only = (
    df_edges_osrm.sort_values("distance_m", ascending=False)[["a","b"]]
    .values.tolist()
)

rank_proposed = (
    df_edges_osrm.sort_values("final_criticality_index", ascending=False)[["a","b"]]
    .values.tolist()
)

K_list = [50, 200]

rows = []
for K in K_list:
    m1 = robustness_metrics_after_removal(G_stop_osrm, rank_osm_only, K)
    rows.append({
        "Approach": "OSM-only (geometry baseline: distance-only)",
        "Edge ranking rule": "sort by distance_m (desc)",
        "K removed": K,
        "R(K) ↑": m1["R(K)"],
        "Components(K) ↓": m1["Components(K)"]
    })

    m2 = robustness_metrics_after_removal(G_stop_osrm, rank_proposed, K)
    rows.append({
        "Approach": "Proposed (OSRM + PT load + GNN)",
        "Edge ranking rule": "sort by final_criticality_index (desc)",
        "K removed": K,
        "R(K) ↑": m2["R(K)"],
        "Components(K) ↓": m2["Components(K)"]
    })

tableX = pd.DataFrame(rows)

pivot = tableX.pivot(index=["Approach","Edge ranking rule"], columns="K removed", values=["R(K) ↑","Components(K) ↓"])
display(pivot)

display(tableX)

import os
os.makedirs("results", exist_ok=True)
tableX.to_csv("results/TableX_OSM_vs_Proposed.csv", index=False)
print("Saved: results/TableX_OSM_vs_Proposed.csv")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def robustness_curve(G_base, edges_ranked, ks):
    rows = []
    for K in ks:
        m = robustness_metrics_after_removal(G_base, edges_ranked, K)
        rows.append({"K_removed": K, "R(K)": m["R(K)"], "Components(K)": m["Components(K)"]})
    return pd.DataFrame(rows)

ks = [0, 10, 25, 50, 100, 200]

rob_osm = robustness_curve(G_stop_osrm, rank_osm_only, ks)
rob_prop = robustness_curve(G_stop_osrm, rank_proposed, ks)

rob_osm["Approach"] = "OSM-only (distance-only)"
rob_prop["Approach"] = "Proposed (final_criticality_index)"

rob_all = pd.concat([rob_osm, rob_prop], ignore_index=True)
display(rob_all)

plt.figure(figsize=(7,4))
plt.plot(rob_osm["K_removed"], rob_osm["R(K)"], marker="o", label="OSM-only (distance)")
plt.plot(rob_prop["K_removed"], rob_prop["R(K)"], marker="o", label="Proposed (OSRM+PT+GNN)")
plt.xlabel("K removed (top-K edges)")
plt.ylabel("R(K) = largest CC ratio")
plt.title("Robustness under targeted edge removal")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7,4))
plt.plot(rob_osm["K_removed"], rob_osm["Components(K)"], marker="o", label="OSM-only (distance)")
plt.plot(rob_prop["K_removed"], rob_prop["Components(K)"], marker="o", label="Proposed (OSRM+PT+GNN)")
plt.xlabel("K removed (top-K edges)")
plt.ylabel("#components(K)")
plt.title("Network fragmentation under targeted edge removal")
plt.legend()
plt.tight_layout()
plt.show()

rob_all.to_csv("results/RobustnessCurves_OSM_vs_Proposed.csv", index=False)
print("Saved: results/RobustnessCurves_OSM_vs_Proposed.csv")

## 13. Visual results


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def short_name(s, max_len=35):
    s = str(s)
    return s if len(s) <= max_len else s[:max_len-1] + "…"

In [ ]:
deg_osrm = np.array([d for _, d in G_stop_osrm.degree()])

plt.figure(figsize=(6,4))
plt.hist(deg_osrm, bins=30)
plt.title("Figure 1. Degree distribution of stops (OSRM stop graph)")
plt.xlabel("Node degree")
plt.ylabel("Count of stops")
plt.tight_layout()
plt.show()

In [ ]:
top15_routes = df_stop.sort_values("routes_cnt", ascending=False).head(15).copy()
top15_routes["name_short"] = top15_routes["name"].map(short_name)

plt.figure(figsize=(9,5))
plt.barh(top15_routes["name_short"], top15_routes["routes_cnt"])
plt.gca().invert_yaxis()
plt.title("Figure 2. Top-15 busiest stops (by number of routes)")
plt.xlabel("Number of routes serving the stop")
plt.ylabel("Stop name")
plt.tight_layout()
plt.show()


In [ ]:
top15_bn = df_stop.sort_values("bottleneck_score", ascending=False).head(15).copy()
top15_bn["name_short"] = top15_bn["name"].map(short_name)

plt.figure(figsize=(9,5))
plt.barh(top15_bn["name_short"], top15_bn["bottleneck_score"])
plt.gca().invert_yaxis()
plt.title("Figure 3. Top-15 bottleneck stops (routes × betweenness)")
plt.xlabel("Bottleneck score (normalized)")
plt.ylabel("Stop name")
plt.tight_layout()
plt.show()


In [ ]:

top10_load = df_edges_osrm.sort_values("load", ascending=False).head(10).copy()

labels = []
for _, r in top10_load.iterrows():
    a_name = stop_id_to_name.get(r["a"], str(r["a"]))
    b_name = stop_id_to_name.get(r["b"], str(r["b"]))
    labels.append(f"{a_name} → {b_name}")

plt.figure(figsize=(10,5))
plt.barh(labels, top10_load["load"])
plt.gca().invert_yaxis()
plt.title("Figure 4. Top-10 PT segments by route load")
plt.xlabel("Route load (# routes using the segment)")
plt.ylabel("Stop-to-stop segment")
plt.tight_layout()
plt.show()

In [ ]:

top10_risk = df_edges_osrm.sort_values("risk", ascending=False).head(10).copy()

labels = []
for _, r in top10_risk.iterrows():
    a_name = stop_id_to_name.get(r["a"], str(r["a"]))
    b_name = stop_id_to_name.get(r["b"], str(r["b"]))
    labels.append(f"{a_name} → {b_name}")

plt.figure(figsize=(10,5))
plt.barh(labels, top10_risk["risk"])
plt.gca().invert_yaxis()
plt.title("Figure 5. Top-10 critical PT segments (load × betweenness × time)")
plt.xlabel("Composite risk (normalized)")
plt.ylabel("Stop-to-stop segment")
plt.tight_layout()
plt.show()

## 14. Folium maps: top-risk segments + top nodes


In [ ]:
import folium
from folium.plugins import MarkerCluster
import numpy as np

df_stop2 = df_stop.dropna(subset=["lat", "lon"]).copy()

q1, q2, q3 = df_stop2["routes_cnt"].quantile([0.50, 0.80, 0.95]).values
print("routes_cnt quantiles:", q1, q2, q3)

def color_routes(cnt):
    if cnt >= q3: return "#8B0000"
    if cnt >= q2: return "#FF0000"
    if cnt >= q1: return "#FF8C00"
    return "#2E86C1"

center_lat = df_stop2["lat"].mean()
center_lon = df_stop2["lon"].mean()
m_over = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles="cartodbpositron")
cluster = MarkerCluster().add_to(m_over)

top = df_stop2.sort_values("routes_cnt", ascending=False).head(400)

for _, r in top.iterrows():
    cnt = int(r["routes_cnt"])
    folium.CircleMarker(
        location=[float(r["lat"]), float(r["lon"])],
        radius=4 + min(cnt, 30) * 0.20,
        color=color_routes(cnt),
        fill=True,
        fill_opacity=0.9,
        popup=f"<b>{r['name']}</b><br>Routes: <b>{cnt}</b><br>ID: {r['stop_id']}"
    ).add_to(cluster)

m_over.save("results/map_overloaded_stops_quantiles.html")
m_over

In [ ]:
import folium
import numpy as np

df_stop2 = df_stop.dropna(subset=["lat","lon"]).copy()

q1, q2, q3 = df_stop2["bottleneck_score"].quantile([0.50, 0.80, 0.95]).values
print("bottleneck_score quantiles:", q1, q2, q3)

def color_bn(s):
    if s >= q3: return "#8B0000"
    if s >= q2: return "#FF0000"
    if s >= q1: return "#FF8C00"
    return "#2E86C1"

center_lat = df_stop2["lat"].mean()
center_lon = df_stop2["lon"].mean()
m_bn = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles="cartodbpositron")

top_bn = df_stop2.sort_values("bottleneck_score", ascending=False).head(300)

for _, r in top_bn.iterrows():
    s = float(r["bottleneck_score"])
    folium.CircleMarker(
        location=[float(r["lat"]), float(r["lon"])],
        radius=5 + 18*s,  # чуть больше
        color=color_bn(s),
        fill=True,
        fill_opacity=0.9,
        popup=(f"<b>{r['name']}</b><br>"
               f"Routes: {int(r['routes_cnt'])}<br>"
               f"Betweenness: {float(r['betw']):.5f}<br>"
               f"Bottleneck score: {s:.3f}")
    ).add_to(m_bn)

m_bn.save("results/map_bottlenecks_quantiles.html")
m_bn

In [ ]:
import folium
import numpy as np

score_col = "final_criticality_index"

q1, q2, q3 = df_edges_osrm[score_col].quantile([0.50, 0.80, 0.95]).values
print(score_col, "quantiles:", q1, q2, q3)

def color_seg(x):
    if x >= q3: return "#8B0000"
    if x >= q2: return "#FF0000"
    if x >= q1: return "#FF8C00"
    return "#2E86C1"

center_lat = df_edges_osrm["a_lat"].mean()
center_lon = df_edges_osrm["a_lon"].mean()
m_seg = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles="cartodbpositron")

top_seg = df_edges_osrm.sort_values(score_col, ascending=False).head(1200)

for _, r in top_seg.iterrows():
    s = float(r[score_col])
    a_name = stop_id_to_name.get(r["a"], str(r["a"]))
    b_name = stop_id_to_name.get(r["b"], str(r["b"]))
    folium.PolyLine(
        locations=[[float(r["a_lat"]), float(r["a_lon"])], [float(r["b_lat"]), float(r["b_lon"])]],
        color=color_seg(s),
        weight=2 + 10*s,
        opacity=0.85,
        popup=(f"<b>Segment:</b> {a_name} → {b_name}<br>"
               f"<b>final:</b> {s:.3f}<br>"
               f"<b>risk:</b> {float(r['risk']):.3f}<br>"
               f"<b>load:</b> {int(r['load'])}<br>"
               f"<b>time:</b> {float(r['travel_time_s']):.1f}s<br>"
               f"<b>speed:</b> {float(r['speed_kmh']):.1f} km/h")
    ).add_to(m_seg)

m_seg.save("results/map_segments_quantiles.html")
m_seg